In [ ]:
import torch
import glob
import os
import librosa
import numpy as np
from tqdm import tqdm

# 前処理済みデータが保存されているディレクトリ
PROCESSED_DIR = "drive/MyDrive/hioki_lab/data/URMP_solo_inst/processed_data"
HOP_LENGTH = 64
N_FFT = 2048

def patch_loudness():
    pt_files = glob.glob(os.path.join(PROCESSED_DIR, "*.pt"))
    print(f"Found {len(pt_files)} files to patch.")

    for path in tqdm(pt_files):
        # 1. 既存データをロード
        data = torch.load(path)
        audio = data['audio'].numpy()

        # 2. A特性なしのRMS音量を計算
        # librosa.feature.rms は [1, T_frames] で返るので、デシベルに変換して整形
        rms = librosa.feature.rms(y=audio, frame_length=N_FFT, hop_length=HOP_LENGTH)
        rms_db = librosa.amplitude_to_db(rms, ref=1.0) # refは前処理の仕様に合わせる

        # 3. フレーム数を f0 等に合わせる (min_len の処理)
        target_len = data['f0'].shape[0]
        rms_db = rms_db[0, :target_len]

        # 4. データを追加して上書き保存
        data['loudness_rms'] = torch.from_numpy(rms_db).float().unsqueeze(-1)

        torch.save(data, path)

if __name__ == "__main__":
    patch_loudness()
    print("Patching completed!")

In [3]:
import torch
import glob
import os
import librosa
import numpy as np
from tqdm import tqdm

# 前処理済みデータが保存されているディレクトリ
PROCESSED_DIR = "drive/MyDrive/hioki_lab/data/URMP_solo_inst/processed_data"
HOP_LENGTH = 64
N_FFT = 2048

pt_files = glob.glob(os.path.join(PROCESSED_DIR, "*.pt"))
print(f"Found {len(pt_files)} files to patch.")
data = torch.load(pt_files[0])
data.keys()


Found 968 files to patch.


dict_keys(['audio', 'mel', 'f0', 'loudness', 'instrument_name', 'loudness_rms'])